# 10 — Synthetic Anomaly Stress Testing

## Objective
Inject controlled anomalies into a separate copy and measure whether the detector ranks known anomalies highly.

> **Scientific contract:** fraud labels are validation ground truth, never unsupervised predictors. Chronological splits protect against future leakage. The test period is locked until Notebook 11.

### Outputs
Reproducible artifacts are saved to `artifacts/` and audit evidence to `reports/`. Interactive controls are for investigation/exploration; the underlying tables remain reproducible.

## 1. Injection library

In [5]:
from pathlib import Path
import sys, json, warnings, numpy as np, pandas as pd
import plotly.express as px
from IPython.display import display, Markdown
import ipywidgets as widgets
warnings.filterwarnings("ignore")
ROOT=Path.cwd()
if not (ROOT/"data").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
from pipeline_utils import *
ART=ROOT/"artifacts"; REP=ROOT/"reports"; ART.mkdir(exist_ok=True); REP.mkdir(exist_ok=True)
RANDOM_STATE=42
fraud,_=load_data(); base=fraud.sample(min(50000,len(fraud)),random_state=42).copy()
def inject_value(df,n=300,mult=10):
 x=df.copy(); rng=np.random.default_rng(42); 
 idx=rng.choice(x.index,min(n,len(x)),replace=False); 
 x["synthetic_anomaly"]=0; 
 x.loc[idx,"purchase_value"]*=mult; x.loc[idx,"synthetic_anomaly"]=1; 
 
 return x
def inject_device(df,n=300):
  x=df.copy(); rng=np.random.default_rng(7);
  idx=rng.choice(x.index,min(n,len(x)),replace=False); 
  x["synthetic_anomaly"]=0; 
  x.loc[idx,"device_id"]="SYNTH_DEVICE"; 
  x.loc[idx,"synthetic_anomaly"]=1; 
  return x


## 2. Stress-test benchmark

In [6]:
cases=[("value_spike",inject_value(base)),("shared_device_burst",inject_device(base))]; rows=[]
for name,x in cases:
 s=abs(robust_z(x.purchase_value))+.25*x.device_id.map(x.device_id.value_counts()).values
 m=ranking_metrics(x.synthetic_anomaly,s); 
 m["scenario"]=name; 
 rows.append(m)
display(pd.DataFrame(rows)); 
pd.DataFrame(rows).to_csv(REP/"synthetic_stress_test.csv",index=False)

,pr_auc,roc_auc,precision_at_50,recall_at_50,precision_at_100,recall_at_100,precision_at_500,recall_at_500,scenario
0,0.979815,0.99976,1.0,0.166667,1.0,0.333333,0.59,0.983333,value_spike
1,1.000000,1.00000,1.0,0.166667,1.0,0.333333,0.60,1.000000,shared_device_burst


## 3. Interactive sensitivity test

In [7]:
mult=widgets.IntSlider(value=10,min=2,max=25,description="Value ×"); n=widgets.IntSlider(value=300,min=50,max=1000,step=50,description="Injected"); out=widgets.Output()
def run(*_):
 x=inject_value(base,n.value,mult.value); 
 s=abs(robust_z(x.purchase_value))
 with out: out.clear_output(); 
 display(pd.DataFrame([ranking_metrics(x.synthetic_anomaly,s,ks=(50,100,500))]))
mult.observe(run,'value'); 
n.observe(run,'value'); 
display(widgets.HBox([mult,n]),out); 
run()

Output()

,pr_auc,roc_auc,precision_at_50,recall_at_50,precision_at_100,recall_at_100,precision_at_500,recall_at_500
0,0.984014,0.999844,1.0,0.166667,1.0,0.333333,0.59,0.983333
